In [4]:
# 1) Монтируем Google Drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# 1) Убедитесь, что Drive примонтирован
from google.colab import drive
drive.mount('/content/drive')

# 2) Посмотрим, что реально лежит в корне MyDrive
import os
print(os.listdir("/content/drive/MyDrive")[:50])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['person_baggage_yolo.zip']


In [7]:
# 2) Путь до папки, где лежит ваш ноутбук и zip (в Colab обычно это /content/drive/MyDrive/...)
from pathlib import Path
import zipfile

# ZIP_PATH = Path("/content/drive/MyDrive/Colab Notebooks/person_baggage_yolo.zip")  # <-- поправьте, если zip в другой папке
# OUT_DIR  = Path("/content/Colab Notebooks/person_baggage_yolo")                   # куда распаковать (в рабочую директорию Colab)

ZIP_PATH = Path("/content/drive/MyDrive/person_baggage_yolo.zip")  # <-- поправьте, если zip в другой папке
OUT_DIR  = Path("/content/person_baggage_yolo")

OUT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(OUT_DIR)

print("Unpacked to:", OUT_DIR)
print("Top-level:", [p.name for p in OUT_DIR.iterdir()])


Unpacked to: /content/person_baggage_yolo
Top-level: ['person_baggage_yolo']


In [8]:
from pathlib import Path

# root = Path("/content/Colab Notebooks/person_baggage_yolo")
root = Path("/content/person_baggage_yolo")
# если внутри есть единственная папка и в ней images/labels — зайдём в неё
subs = [p for p in root.iterdir() if p.is_dir()]
if not (root/"images").exists() and len(subs) == 1 and (subs[0]/"images").exists():
    root = subs[0]

print("Dataset root:", root)
print("Has images:", (root/"images").exists(), "Has labels:", (root/"labels").exists())


Dataset root: /content/person_baggage_yolo/person_baggage_yolo
Has images: True Has labels: True


In [9]:
from pathlib import Path

data_yaml = root / "data.yaml"
data_yaml.write_text(f"""path: {root}
train: images/train
val: images/val
test: images/test

names:
  0: person
  1: baggage
""")
print("Wrote:", data_yaml)


Wrote: /content/person_baggage_yolo/person_baggage_yolo/data.yaml


In [10]:
from tqdm import tqdm

# =========================
# 2) Находим zip на Google Drive и распаковываем
# =========================
ZIP_GLOB = "person_baggage_yolo*.zip"  # если имя чуть отличается — всё равно найдёт
drive_root = Path("/content/drive/MyDrive")

matches = list(drive_root.rglob(ZIP_GLOB))
assert len(matches) > 0, f"Не найден zip по шаблону {ZIP_GLOB} в {drive_root}"

zip_path = matches[0]
print("ZIP found:", zip_path)

out_dir = Path("/content/person_baggage_yolo")  # распаковка в рабочую директорию Colab
out_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    members = z.infolist()
    for m in tqdm(members, desc="Unzipping", unit="file"):
        z.extract(m, out_dir)

print("Unpacked to:", out_dir)
print("Top-level:", [p.name for p in out_dir.iterdir()])


ZIP found: /content/drive/MyDrive/person_baggage_yolo.zip


Unzipping: 100%|██████████| 20010/20010 [00:21<00:00, 925.59file/s]

Unpacked to: /content/person_baggage_yolo
Top-level: ['person_baggage_yolo']


In [11]:
# =========================
# 3) Определяем корневую папку датасета (учёт лишней вложенности)
# =========================
root = out_dir

# если внутри одна папка и там images/labels — используем её как root
subs = [p for p in root.iterdir() if p.is_dir()]
if not (root / "images").exists() and len(subs) == 1 and (subs[0] / "images").exists():
    root = subs[0]

assert (root / "images").exists(), f"Нет папки images в {root}"
assert (root / "labels").exists(), f"Нет папки labels в {root}"

print("Dataset root:", root)

for split in ["train", "val", "test"]:
    img_dir = root / "images" / split
    lab_dir = root / "labels" / split
    if img_dir.exists():
        print(f"{split:5} images:", len(list(img_dir.glob("*"))))
    if lab_dir.exists():
        print(f"{split:5} labels:", len(list(lab_dir.glob("*.txt"))))


Dataset root: /content/person_baggage_yolo/person_baggage_yolo
train images: 8000
train labels: 8000
val   images: 1000
val   labels: 1000
test  images: 1000
test  labels: 1000


In [12]:
# =========================
# 4) Создаём data.yaml
#    ВАЖНО: names должны соответствовать id классов в ваших .txt (0,1,...)
# =========================
import yaml

data_yaml = root / "data.yaml"

data = {
    "path": str(root),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",  # если test нет — строку можно оставить, Ultralytics обычно просто проигнорирует
    "names": {
        0: "person",
        1: "baggage",
    },
}

with open(data_yaml, "w", encoding="utf-8") as f:
    yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)

print("Wrote:", data_yaml)
print(open(data_yaml, "r", encoding="utf-8").read())


Wrote: /content/person_baggage_yolo/person_baggage_yolo/data.yaml
path: /content/person_baggage_yolo/person_baggage_yolo
train: images/train
val: images/val
test: images/test
names:
  0: person
  1: baggage



In [13]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


torch: 2.9.0+cu126
cuda available: True
gpu: Tesla T4


In [14]:
!pip -q install -U ultralytics tqdm pyyaml pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 147.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.0 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.0 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.0 which is incompatible.
prophet 1.2.2 requires pandas<3,>=1.0.4, but you have pandas 3.0.0 which is incompatible.
cudf-cu12 25.10.0 requires pandas<2.4.0dev0,>=2.0, but you have pandas 3.0.0 which is incompatible.
dask-cudf-cu12 25.10.0 requires pandas<2.4.0dev0,>=2.0, but you have pandas 3.0.0 which is incompatible.
bqplot 0.12.45 

In [16]:
from ultralytics import YOLO
from pathlib import Path
from tqdm.auto import tqdm
import torch, json
import pandas as pd

# путь к вашему data.yaml
data_yaml = "/content/person_baggage_yolo/person_baggage_yolo/data.yaml"  # поправьте при необходимости
assert Path(data_yaml).exists(), f"Not found: {data_yaml}"

# авто-выбор устройства
device = 0 if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# адекватные дефолты под CPU
EPOCHS = 50
IMGSZ  = 640
BATCH  = 16 if device != "cpu" else 4
WORKERS = 4 if device != "cpu" else 2

model = YOLO("yolo11s.pt")

# ---------- tqdm progressbar через callbacks ----------
state = {"pbar": None}

def on_train_start(trainer):
    state["pbar"] = tqdm(total=trainer.args.epochs, desc="Training (epochs)", unit="epoch")

def on_train_epoch_end(trainer):
    # шаг прогресса
    state["pbar"].update(1)

    # аккуратно вытаскиваем метрики, если они доступны в этой версии ultralytics
    postfix = {}
    m = getattr(trainer, "metrics", None)
    if isinstance(m, dict):
        for k, short in [
            ("metrics/precision(B)", "P"),
            ("metrics/recall(B)", "R"),
            ("metrics/mAP50(B)", "mAP50"),
            ("metrics/mAP50-95(B)", "mAP50-95"),
        ]:
            if k in m:
                try:
                    postfix[short] = float(m[k])
                except:
                    pass
    if postfix:
        state["pbar"].set_postfix(postfix)

def on_train_end(trainer):
    if state["pbar"] is not None:
        state["pbar"].close()

model.add_callback("on_train_start", on_train_start)
model.add_callback("on_train_epoch_end", on_train_epoch_end)
model.add_callback("on_train_end", on_train_end)

# ---------- train ----------
train_results = model.train(
    data=data_yaml,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=device,
    workers=WORKERS,
    patience=20,
    project="runs",
    name="person_baggage_yolo11",
)

best_pt = Path(model.trainer.best)
print("Best weights:", best_pt)

# ---------- metrics ----------
best_model = YOLO(str(best_pt))

def eval_split(split: str):
    r = best_model.val(
        data=data_yaml,
        split=split,
        imgsz=IMGSZ,
        device=device,
        conf=0.001,  # стандартно для mAP ставят низкий conf
        iou=0.6
    )
    return {
        "split": split,
        "precision": float(r.box.mp),
        "recall": float(r.box.mr),
        "mAP50": float(r.box.map50),
        "mAP50_95": float(r.box.map),
    }

metrics = {"val": eval_split("val")}

# test опционален
ds_root = Path(data_yaml).parent
if (ds_root / "images" / "test").exists():
    metrics["test"] = eval_split("test")

# сохраняем в JSON + CSV
Path("metrics_person_baggage_yolo11.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
df = pd.DataFrame(list(metrics.values()))
df.to_csv("metrics_person_baggage_yolo11.csv", index=False)

print(metrics)
df


Using device: 0
Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/person_baggage_yolo/person_baggage_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=person_baggage_yolo112, nbs=64, nms=False, opset=None, optimize=False, opt

Training (epochs):   0%|          | 0/50 [00:00<?, ?epoch/s]

Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /content/runs/detect/runs/person_baggage_yolo112
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50       5.7G      1.469      1.678      1.401        174        640: 100% ━━━━━━━━━━━━ 500/500 2.8it/s 2:59
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 3.4it/s 9.3s
                   all       1000       4461      0.335      0.193      0.193     0.0936

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50       5.7G      1.702      1.803      1.575        184        640: 100% ━━━━━━━━━━━━ 500/500 2.9it/s 2:52
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 3.4it/s 9.4s
                   all       1000       4461      0.283      0.249       0.19     0.0833

      Epoch 

,split,precision,recall,mAP50,mAP50_95
0,val,0.559354,0.486502,0.483534,0.276235
1,test,0.567563,0.473541,0.483332,0.273217


In [18]:
from pathlib import Path
from google.colab import files

best_pt = Path("/content/runs/detect/runs/person_baggage_yolo112/weights/best.pt")
assert best_pt.exists(), best_pt

files.download(str(best_pt))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
from google.colab import drive
drive.mount("/content/drive")

import shutil
from pathlib import Path

best_pt = Path("/content/runs/detect/runs/person_baggage_yolo112/weights/best.pt")
dst = Path("/content/drive/MyDrive/person_baggage_best.pt")
shutil.copy2(best_pt, dst)
print("Saved to:", dst)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to: /content/drive/MyDrive/person_baggage_best.pt
